# Caracal RL cyber (Kaggle T4) — GRPO e RSI separados + benchmark final completo

**Settings**: Accelerator -> **GPU T4 x2** · Internet **ON** · Save Version -> Run All (batch).

Um Run All faz, em sequência:
1. mede a **base** (Qwen2.5-Coder-3B, sem adapter) no dev
2. **GRPO** direto → **benchmark do GRPO** logo depois (curva de performance por step)
3. **RSI** outer loop (2×2×10) → **performance por geração** logo depois (trajetória)
4. **BENCHMARK FINAL** — a suíte cyber inteira (cti_bench, cybermetric, secqa,
   secbench, mmlu_security, seceval, cybersoceval) no base vs melhor GRPO vs
   melhor RSI, lado a lado. Salva `final_benchmark.json`.

Os passos 2/3 medem no dev do CVE→CWE (o alvo do RL). O passo 4 mede a suíte
toda pra ver se o ganho no alvo custou regressão nos outros benches.

**Régua**: base ≈ 44% no CVE→CWE. Piso de colapso (sempre CWE-79) = 0.273.
Alvo = Foundation-Sec-8B RCM 72–75.

**Quota**: cada conta Kaggle tem 30h/semana de GPU. Se a sua esgotou, rode em
outra conta de founder — clona tudo do GitHub. A suíte final é pesada (~3 modelos
× 7 benches); reduza os `n` em `KW` se aproximar das 12h.

In [ ]:
# Roda GRPO e RSI no mesmo run, benchmarka os dois no fim contra a base.
GRPO_STEPS = 80             # RL direto, eval a cada 20 (curva)
RSI_GENS, RSI_CANDS, RSI_STEPS = 2, 2, 10   # outer loop: 2x2x10 steps/cand
BASE = 'Qwen/Qwen2.5-Coder-3B-Instruct'
print(f'GRPO {GRPO_STEPS} steps + RSI {RSI_GENS}x{RSI_CANDS}x{RSI_STEPS}, benchmark dos 2 no fim')

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'Sem GPU: Settings -> Accelerator -> GPU T4 x2'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv']).decode())

In [ ]:
# Stack pinado (torch 2.6 cu124, sem Unsloth — T4 e SM 7.5). torchao removido.
!pip -q install 'torch==2.6.0' 'torchvision==0.21.0' --index-url https://download.pytorch.org/whl/cu124
!pip -q install 'transformers==4.49.0' 'peft==0.14.0' 'trl==0.15.2' 'accelerate>=1.0.0' 'datasets>=3.0.0' 'sentence-transformers' 'scipy' 'statsmodels' 'antlr4-python3-runtime==4.11'
!pip -q uninstall -y torchao 2>/dev/null
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# Codigo com todos os fixes + dataset CVE->CWE + primitivo de benchmark
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git','clone','--depth','1','-b','s07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
print(subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
subprocess.run(['python','-m','data.ignite.build_cyber_rcm'], check=True)

import glob
# mesmo primitivo que os drivers usam (uma copia de load->bench->free)
from eval.ignite.benches.run import bench_adapter, load_eval_model, free_gpu

DEV = 'data/ignite/cyber_rcm_dev.jsonl'

def bench(adapter=None):
    r = bench_adapter(BASE, adapter, 'cyber_rcm', DEV, 150)
    return r['accuracy'], r['hier_score'], r['unparsed_frac']

def has_adapter(d):
    return bool(glob.glob(d+'/adapter_*.safetensors') or glob.glob(d+'/adapter_*.bin'))

# base medida uma vez, e a referencia dos dois benchmarks abaixo
BASE_ACC, BASE_H, BASE_U = bench(None)
print(f'BASE dev acc={BASE_ACC:.3f} hier={BASE_H:.3f} unparsed={BASE_U:.2f}  (piso de colapso 0.273)')

In [ ]:
# ===== GRPO direto (RL puro) + benchmark logo depois =====
import sys, os, time, json
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
OUT_GRPO = '/kaggle/working/rl_cyber_grpo'

t=time.time()
grpo = [sys.executable,'-m','train.ignite.B_grpo_straight',
        '--base',BASE,'--out',OUT_GRPO,'--total-steps',str(GRPO_STEPS),
        '--block','20','--lr','1e-6','--rank','32','--eval-n','150']
if os.path.exists(f'{OUT_GRPO}/curve.json'): grpo.append('--resume')
subprocess.run(grpo, check=True, env=env)
print(f'GRPO treinou em {(time.time()-t)/60:.0f} min', flush=True)

print('\n=== BENCHMARK GRPO (dev n=150 | base={:.3f} | piso 0.273 | alvo 72-75) ==='.format(BASE_ACC))
curve = json.load(open(f'{OUT_GRPO}/curve.json'))
for s,v in sorted(curve.items(), key=lambda x:int(x[0])):
    d = v['accuracy']-BASE_ACC
    print(f"  step {int(s):3d}: acc={v['accuracy']:.3f} hier={v['hier_score']:.3f} unparsed={v['unparsed_frac']:.2f}  ({d:+.3f} vs base)")
grpo_best = max(v['accuracy'] for v in curve.values())
print(f"\nGRPO melhor={grpo_best:.3f}  (delta vs base {grpo_best-BASE_ACC:+.3f})")

In [ ]:
# ===== RSI outer loop (same-model) + benchmark logo depois =====
OUT_RSI = '/kaggle/working/rl_cyber_rsi'
t=time.time()
rsi = [sys.executable,'-m','train.ignite.C_rsi_outer',
       '--base',BASE,'--bench','cyber_rcm','--bench-name','cyber_rcm',
       '--dataset-train','data/ignite/cyber_rcm_train.jsonl',
       '--dataset-dev','data/ignite/cyber_rcm_dev.jsonl',
       '--dataset-val','data/ignite/cyber_rcm_val.jsonl',
       '--gens',str(RSI_GENS),'--cands',str(RSI_CANDS),'--steps',str(RSI_STEPS),'--out',OUT_RSI]
subprocess.run(rsi, check=True, env=env)
print(f'RSI treinou em {(time.time()-t)/60:.0f} min', flush=True)

# Trajetoria de PERFORMANCE a cada melhoria (gravada pelo outer loop por geracao)
print('\n=== PERFORMANCE RSI a cada geracao (dev n=150 | piso 0.273 | alvo 72-75) ===')
traj = json.load(open(f'{OUT_RSI}/trajectory.json'))
b = next((p for p in traj if p['gen']==-1), traj[0])['accuracy']
for p in traj:
    ret = {True:'RETIDO', False:'rejeit', None:'-'}[p['retained']]
    print(f"  {p['label']:6s} acc={p['accuracy']:.3f} hier={p['hier_score']:.3f} "
          f"unparsed={p['unparsed_frac']:.2f}  ({p['accuracy']-b:+.3f} vs base) [{ret}]")
final = traj[-1]['accuracy']
print(f"\nbase={b:.3f} -> final RSI={final:.3f}  (delta {final-b:+.3f})")

In [ ]:
# ===== BENCHMARK FINAL COMPLETO: suite inteira, base vs melhor GRPO vs melhor RSI =====
import json, glob
from eval.s07.benches import BENCH_REGISTRY as S07

def grpo_best_adapter():
    if not os.path.exists(f'{OUT_GRPO}/curve.json'): return None
    cur = json.load(open(f'{OUT_GRPO}/curve.json'))
    step = max(cur, key=lambda s: cur[s]['accuracy'])
    d = f'{OUT_GRPO}/block-{step}'
    return d if has_adapter(d) else None

def rsi_best_adapter():
    sp = f'{OUT_RSI}/state.json'
    if os.path.exists(sp):
        v = json.load(open(sp)).get('v_adapter')
        if v and has_adapter(v): return v
    return None   # nada retido -> RSI = base

MODELS = {'base': None, 'GRPO': grpo_best_adapter(), 'RSI': rsi_best_adapter()}

# suite completa (mesmos benches/n do paper). Reduza os n se estourar as 12h.
KW = {'cti_bench':{'n_rcm':500,'n_mcq':500,'subsets':['rcm','mcq']}, 'cybermetric':{'tier':500},
      'secqa':{}, 'secbench':{'n_mcq':300}, 'mmlu_security':{'n':100}, 'seceval':{'n':200},
      'cybersoceval':{'n':100}}

def run_suite(adapter):
    m, tk = load_eval_model(BASE, adapter)   # mesmo loader dos drivers
    out = {}
    for name in KW:
        try:
            r = S07[name](m, tk, **KW[name])
            if 'accuracy' in r: out[name] = r['accuracy']
            else:
                for sub, sv in r.items():
                    if isinstance(sv, dict) and 'accuracy' in sv: out[f'{name}.{sub}'] = sv['accuracy']
        except Exception as e:
            out[name] = f'ERR {str(e)[:40]}'
    m = tk = None; free_gpu()
    return out

print('=== BENCHMARK FINAL (suite completa) ===', flush=True)
res = {}
for tag, ad in MODELS.items():
    print(f'  rodando {tag} (adapter={ad})...', flush=True)
    res[tag] = run_suite(ad)
    json.dump(res, open('/kaggle/working/final_benchmark.json','w'), indent=2)

benches = sorted({k for v in res.values() for k in v})
print(f"\n{'bench':24s} {'base':>8s} {'GRPO':>8s} {'RSI':>8s}")
for bch in benches:
    row = f"{bch:24s}"
    for tag in ('base','GRPO','RSI'):
        v = res[tag].get(bch)
        row += f" {v*100:7.1f}" if isinstance(v,(int,float)) else f" {'--':>7s}"
    print(row)
print('\nsalvo em /kaggle/working/final_benchmark.json')